In [ ]:
!wget -q "https://www.timeseriesclassification.com/aeon-toolkit/FordA.zip"
!unzip -q FordA.zip

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
import time
import random

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [ ]:
train = pd.read_csv("FordA_TRAIN.txt", header=None, sep=r'\s+')
test  = pd.read_csv("FordA_TEST.txt",  header=None, sep=r'\s+')

X_train = train.iloc[:, 1:].values.astype(np.float32)
y_train = train.iloc[:, 0].values
X_test  = test.iloc[:, 1:].values.astype(np.float32)
y_test  = test.iloc[:, 0].values

le = LabelEncoder()
y_train = le.fit_transform(y_train).astype(np.int64)
y_test  = le.transform(y_test).astype(np.int64)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Sequence length: {X_train.shape[1]}")
print(f"Classes: {len(np.unique(y_train))}")
print(f"Class distribution (train): {np.bincount(y_train)}")
print(f"Class distribution (test): {np.bincount(y_test)}")

Expanded Train: (12444, 1, 405)
Expanded Test:  (12505, 1, 405)
Class distribution: [8967 3477]
Input shape: torch.Size([12444, 1, 405])


In [ ]:
mean = X_train.mean(axis=1, keepdims=True)
std  = X_train.std(axis=1, keepdims=True) + 1e-8
X_train_norm = (X_train - mean) / std

mean = X_test.mean(axis=1, keepdims=True)
std  = X_test.std(axis=1, keepdims=True) + 1e-8
X_test_norm = (X_test - mean) / std

X_train_t = torch.tensor(X_train_norm).unsqueeze(1)  # (batch, 1, 500)
X_test_t  = torch.tensor(X_test_norm).unsqueeze(1)
y_train_t = torch.tensor(y_train)
y_test_t  = torch.tensor(y_test)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=32, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test_t, y_test_t),  batch_size=32, shuffle=False)

print(f"Input shape: {X_train_t.shape}")

In [ ]:
class CNN_TemporalPooling(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2),

            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2, stride=2),

            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        self.gap = nn.AdaptiveAvgPool1d(1)
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.network(x)
        x = self.gap(x).squeeze(-1)
        return self.classifier(x)


class CNN_LSTM(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(1, 32, kernel_size=7, padding=3),
            nn.ReLU(),
            nn.Conv1d(32, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
        )
        self.lstm = nn.LSTM(input_size=128, hidden_size=128,
                            num_layers=2, batch_first=True)
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.cnn(x)
        x = x.permute(0, 2, 1)
        _, (h_n, _) = self.lstm(x)
        return self.classifier(h_n[-1])

In [ ]:
def train_model_es(model, train_loader, test_loader, epochs=50, lr=1e-3, patience=8):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    train_losses, test_accs = [], []
    best_acc = 0.0
    patience_counter = 0
    best_epoch = 0
    start = time.time()

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0
        for X_batch, y_batch in train_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        model.eval()
        preds, labels = [], []
        with torch.no_grad():
            for X_batch, y_batch in test_loader:
                pred = model(X_batch.to(device)).argmax(dim=1).cpu().numpy()
                preds.extend(pred)
                labels.extend(y_batch.numpy())

        acc = accuracy_score(labels, preds)
        train_losses.append(epoch_loss / len(train_loader))
        test_accs.append(acc)

        if acc > best_acc:
            best_acc = acc
            patience_counter = 0
            best_epoch = epoch + 1
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {epoch+1}, best epoch was {best_epoch}")
                break

        if (epoch+1) % 5 == 0:
            print(f"Epoch {epoch+1} | Loss: {train_losses[-1]:.4f} | Acc: {acc:.4f}")

    training_time = time.time() - start
    return train_losses, test_accs, training_time, best_acc, best_epoch

In [ ]:
import pickle

NUM_CLASSES = len(np.unique(y_train))
seeds = [123, ]
baseline_results = []

for seed in seeds:
    print(f"\n{'='*50}\nSEED {seed}\n{'='*50}")

    set_seed(seed)
    print("Training CNN + Temporal Pooling...")
    pool_model = CNN_TemporalPooling(NUM_CLASSES)
    pool_losses, pool_accs, pool_time, pool_best, pool_epoch = train_model_es(
        pool_model, train_loader, test_loader)

    set_seed(seed)
    print("\nTraining CNN + LSTM...")
    lstm_model = CNN_LSTM(NUM_CLASSES)
    lstm_losses, lstm_accs, lstm_time, lstm_best, lstm_epoch = train_model_es(
        lstm_model, train_loader, test_loader)

    baseline_results.append({
        'seed': seed,
        'pool_acc': pool_best, 'pool_time': pool_time, 'pool_epoch': pool_epoch,
        'lstm_acc': lstm_best, 'lstm_time': lstm_time, 'lstm_epoch': lstm_epoch,
    })

    print(f"\nSeed {seed} | CNN+Pooling: {pool_best:.4f} ({pool_time:.1f}s) | CNN+LSTM: {lstm_best:.4f} ({lstm_time:.1f}s)")

baseline_df = pd.DataFrame(baseline_results)
baseline_df.to_csv('forda_baseline.csv', index=False)
print(baseline_df)

In [ ]:
lengths = [100, 200, 300, 400, 500]
length_results = []

for length in lengths:
    print(f"\n{'='*50}\nSEQUENCE LENGTH {length}\n{'='*50}")

    X_train_short = X_train_t[:, :, :length]
    X_test_short  = X_test_t[:, :, :length]

    train_loader_s = DataLoader(TensorDataset(X_train_short, y_train_t), batch_size=32, shuffle=True)
    test_loader_s  = DataLoader(TensorDataset(X_test_short, y_test_t),  batch_size=32, shuffle=False)

    for seed in seeds:
        set_seed(seed)
        model_pool = CNN_TemporalPooling(NUM_CLASSES)
        _, _, pool_time, pool_acc, _ = train_model_es(model_pool, train_loader_s, test_loader_s)

        set_seed(seed)
        model_lstm = CNN_LSTM(NUM_CLASSES)
        _, _, lstm_time, lstm_acc, _ = train_model_es(model_lstm, train_loader_s, test_loader_s)

        length_results.append({
            'seed': seed, 'length': length,
            'pool_acc': pool_acc, 'pool_time': pool_time,
            'lstm_acc': lstm_acc, 'lstm_time': lstm_time,
        })

        print(f"Seed {seed} | Length {length} | Pool: {pool_acc:.4f} | LSTM: {lstm_acc:.4f}")

length_df = pd.DataFrame(length_results)
length_df.to_csv('forda_sequence_length.csv', index=False)
print(length_df)

Training CNN + Temporal Pooling...
Epoch 5 | Loss: 0.6589 | Acc: 0.5738
Epoch 10 | Loss: 0.6488 | Acc: 0.5643
Epoch 15 | Loss: 0.6442 | Acc: 0.5559
Epoch 20 | Loss: 0.6388 | Acc: 0.5802
Epoch 25 | Loss: 0.6346 | Acc: 0.5440
Epoch 30 | Loss: 0.6283 | Acc: 0.5914
Epoch 35 | Loss: 0.6227 | Acc: 0.6337
Epoch 40 | Loss: 0.6179 | Acc: 0.5274
Epoch 45 | Loss: 0.6132 | Acc: 0.6145
Epoch 50 | Loss: 0.6079 | Acc: 0.5713

Training CNN + LSTM...
Epoch 5 | Loss: 0.6523 | Acc: 0.5104
Epoch 10 | Loss: 0.5935 | Acc: 0.5569


In [ ]:
summary = length_df.groupby('length')[['pool_acc', 'lstm_acc']].mean()

plt.figure(figsize=(8, 5))
plt.plot(summary.index, summary['pool_acc'], marker='o', label='CNN+Pooling')
plt.plot(summary.index, summary['lstm_acc'], marker='o', label='CNN+LSTM')
plt.xlabel('Sequence Length')
plt.ylabel('Test Accuracy')
plt.title('FordA — Accuracy vs Sequence Length (Mean across 3 seeds)')
plt.legend()
plt.grid(True)
plt.show()

print(summary)


STARTING MULTI-LENGTH EVALUATION (USING TRUNCATION)

Testing sequence length: 160
Truncating from 405 to 160 timesteps...
Truncated shape - Train: torch.Size([204, 1, 160]), Test: torch.Size([205, 1, 160])

Training CNN + Temporal Pooling for length 160...
Epoch 5 | Loss: 0.6294 | Acc: 0.7220
Epoch 10 | Loss: 0.5838 | Acc: 0.7220
Early stopping at epoch 11, best epoch was 1

Training CNN + LSTM for length 160...
Epoch 5 | Loss: 0.6103 | Acc: 0.7220
Epoch 10 | Loss: 0.5763 | Acc: 0.7220
Early stopping at epoch 11, best epoch was 1

Results for length 160:
CNN+Pooling | Acc: 0.7220 | Best Epoch: 1 | Time: 1.2s
CNN+LSTM    | Acc: 0.7220 | Best Epoch: 1 | Time: 17.5s

Testing sequence length: 240
Truncating from 405 to 240 timesteps...
Truncated shape - Train: torch.Size([204, 1, 240]), Test: torch.Size([205, 1, 240])

Training CNN + Temporal Pooling for length 240...
Epoch 5 | Loss: 0.6294 | Acc: 0.7220
Epoch 10 | Loss: 0.5840 | Acc: 0.7220
Early stopping at epoch 11, best epoch was 1

T